# Closed-loop smoke (remote-friendly)

Proves **command → FB** on one actuator slot without product teleop.
Safe for offsite review: print a JSON summary; keep `DELTA` tiny; blank when done.

One COM owner — close ``pcb_lab.debug test`` / dashboard first.

In [ ]:
import json
import os
import sys
import time

cwd = os.path.abspath(os.getcwd())
if os.path.basename(cwd) == "notebooks":
    SCRIPTS = os.path.abspath(os.path.join(cwd, ".."))
elif os.path.isdir(os.path.join(cwd, "deft_controls_sdk")):
    SCRIPTS = cwd
else:
    SCRIPTS = os.path.abspath(os.path.join(cwd, "scripts"))
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)

from deft_controls_sdk import HostProxy
from deft_controls_sdk.link.exchange import list_serial_ports
from deft_controls_sdk.config import single_profile, assembly_from_name
from deft_controls_sdk.debug import as_hex

list_serial_ports()
PORT = "COM5"  # set from list above
SLOT = 22
DELTA = 0.05  # rad — keep small
HOLD_S = 1.0

## Connect + discover

In [ ]:
asm = assembly_from_name("bench")  # optional named map; omit for slot-only
proxy = HostProxy.connect(
    PORT, mode="debug", armed=False, assembly=asm
)
hub = proxy.hub
print({"ok": proxy.doctor().get("ok"), "port": hub.port, "mode": proxy.mode})
print("cfg enabled", proxy.cfg_snapshot().get("enabled_count"))

rs = hub.debug.discover_robstride_by_bus(buses=[5, 6], start=0x70, end=0x75)
print("RS", as_hex(rs))

## Optional CFG for the slot under test

Edit bus / protocol / motor_id to match discover hits.

In [ ]:
prof = single_profile(
    SLOT,
    protocol="robstride",
    motor_id=0x70,
    bus=5,
    name=f"slot_{SLOT}",
)
for row in prof.as_cfg_rows():
    print(hub.debug.cfg_set_slot(**row, persist=False))

## Closed-loop: mount/apply hold → nudge → clear

In [ ]:
proxy.arm_plant()
time.sleep(0.1)

a = proxy.actions
view = a.actuator(slots=(SLOT,), name=f"slot_{SLOT}")

before = view.positions()
assert before is not None, "need FB before hold/nudge"
a.mount(view.hold())  # sample FB → stay put
a.apply()
time.sleep(HOLD_S)

a.mount(view.nudge(index=0, delta=DELTA))
print("pending", a.pending)
a.apply()
time.sleep(HOLD_S)

after = view.positions() or list(before)
delta_meas = float(after[0]) - float(before[0])
moved = abs(delta_meas) >= abs(DELTA) * 0.25  # 25% of commanded delta

a.clear()
proxy.disarm_plant()

summary = {
    "ok": bool(moved),
    "slot": SLOT,
    "before": before,
    "after": after,
    "commanded_delta": DELTA,
    "measured_delta": delta_meas,
    "note": "command→FB smoke; not product teleop",
}
print(json.dumps(summary, indent=2))
assert summary["ok"], summary

In [ ]:
proxy.close()